In [ ]:
import os
import shutil
import torch
from torchvision.utils import save_image
from torch.utils.data import DataLoader, Subset

# ==========================================
# 1. CLEAN ENVIRONMENT SETUP
# ==========================================
os.chdir('/kaggle/working')
if os.path.exists('Patch-DM'): shutil.rmtree('Patch-DM')

!git clone https://github.com/mlpc-ucsd/Patch-DM.git
os.chdir('Patch-DM')
!pip install -q pytorch-lightning==1.9.5 lmdb==1.2.1 lpips pytorch-fid einops
!pip install -q git+https://github.com/openai/CLIP.git

# ==========================================
# 2. SURGICAL ARCHITECTURE PATCHES
# ==========================================
with open('experiment.py', 'r') as f: exp = f.read()
with open('experiment.py', 'w') as f: f.write(exp.replace('from numpy.lib.function_base import flip', 'from numpy import flip'))

# Inject your GatedWeightBlender
with open('model/unet.py', 'r') as f: unet = f.readlines()
blender_class = [
    "\nclass GatedWeightBlender(nn.Module):\n",
    "    def __init__(self, channels):\n",
    "        super().__init__()\n",
    "        self.gate = nn.Sequential(\n",
    "            nn.Conv2d(channels, channels // 2, kernel_size=1),\n",
    "            nn.SiLU(),\n",
    "            nn.Conv2d(channels // 2, 1, kernel_size=1),\n",
    "            nn.Sigmoid()\n",
    "        )\n",
    "    def forward(self, x_orig, x_shifted):\n",
    "        alpha = self.gate(x_orig)\n",
    "        return alpha * x_orig + (1 - alpha) * x_shifted\n\n"
]
unet.insert(15, "".join(blender_class))
unet_content = "".join(unet)
init_anchor = "if conf.resnet_use_zero_module:"
unet_content = unet_content.replace(init_anchor, f"self.blender = GatedWeightBlender(ch)\n        {init_anchor}", 1)
forward_anchor = "pred = self.out(h)"
blending_logic = "h_blended = self.blender(h, torch.roll(h, shifts=(2,2), dims=(2,3)))\n        pred = self.out(h_blended)"
with open('model/unet.py', 'w') as f: f.write(unet_content.replace(forward_anchor, blending_logic, 1))

# ==========================================
# 3. FIX SEMANTIC PATH (Crucial for Encoder)
# ==========================================
INPUT_SEMANTIC = "/kaggle/input/datasets/vanditagupta2609/lmdb-input-2/lmdb-input/semantic.pt"
FIXED_SEMANTIC_PATH = '/kaggle/working/semantic_fixed.pt'
if os.path.exists(INPUT_SEMANTIC):
    data = torch.load(INPUT_SEMANTIC, map_location='cpu')
    wrapped = {'model_state_dict': {'semantic_enc.weight': data[list(data.keys())[0]]}} if isinstance(data, dict) and 'model_state_dict' not in data else data
    torch.save(wrapped, FIXED_SEMANTIC_PATH)

# ==========================================
# 4. LOAD FULL 214M MODEL
# ==========================================
from utils.dataset import FFHQlmdb
from templates import train_autoenc
from experiment import LitModel

CKPT_PATH = "/kaggle/input/datasets/vanditagupta2609/novelty-2/Patch-DM/checkpoints/patchdm_novelty/last.ckpt"

print("🧠 Loading architecture and weights...")
conf = train_autoenc()
conf.batch_size = 4
conf.patch_size = 64
conf.semantic_path = FIXED_SEMANTIC_PATH

model = LitModel.load_from_checkpoint(CKPT_PATH, conf=conf, strict=False).cuda()
model.eval()

# ==========================================
# 5. PREPARE 100 TARGET IMAGES
# ==========================================
print("\n1️⃣ Extracting 100 Real Target Images...")
os.makedirs('/kaggle/working/real_images', exist_ok=True)
dataset = FFHQlmdb(path="/kaggle/input/datasets/vanditagupta2609/lmdb-input-2/lmdb-input/ffhq256.lmdb", image_size=256)
dataloader = DataLoader(Subset(dataset, range(100)), batch_size=4, shuffle=False)

for i in range(100):
    save_image((dataset[i]['img'] + 1) / 2, f'/kaggle/working/real_images/{i}.png')

# ==========================================
# 6. GENERATE USING NATIVE DIFFUSION LOOP
# ==========================================
print("\n2️⃣ Generating 100 Fake Images...")
os.makedirs('/kaggle/working/fake_images', exist_ok=True)

sampler = conf._make_diffusion_conf(conf.T_eval).make_sampler()
global_idx = 0

with torch.no_grad():
    for batch in dataloader:
        real_imgs = batch['img'].cuda()
        b_size = real_imgs.shape[0]
        
        # 1. Semantic Lookup
        batch_indices = torch.arange(global_idx, global_idx + b_size, dtype=torch.long, device='cuda')
        cond = model.ema_model.encoder(batch_indices)
        
        # 2. Construct the Positional Grid
        # Because Patch-DM pads 32 pixels on all sides, a 256x256 image becomes 320x320.
        # 320 / 64 = 5 patches per dimension (25 patches total per image).
        pos_y, pos_x = torch.meshgrid(torch.arange(5), torch.arange(5), indexing='ij')
        pos = torch.stack([pos_y, pos_x], dim=-1).reshape(-1, 2).cuda().repeat(b_size, 1)
        
        # 3. Explicitly construct the exact kwargs the loop needs
        model_kwargs = {
            'cond': cond,
            'imgs': real_imgs, 
            'x_start': real_imgs
        }
        
        # 4. Call the sampler. This triggers p_sample_loop_progressive which 
        #    handles ALL the CFG multiplication, patchification, and stitching automatically.
        gen_imgs = sampler.sample(
            model=model.ema_model,
            shape=real_imgs.shape,
            noise=torch.randn_like(real_imgs),
            all_pos=[pos],
            model_kwargs=model_kwargs
        )
        
        # 5. Save to disk
        gen_imgs = (gen_imgs + 1) / 2
        for j in range(b_size):
            if global_idx + j < 100:
                save_image(gen_imgs[j], f'/kaggle/working/fake_images/{global_idx + j}.png')
            
        global_idx += b_size
        print(f"Generated {global_idx}/100 images...")

# ==========================================
# 7. CALCULATE FID & ZIP
# ==========================================
print("\n📊 FID SCORE:")
!python -m pytorch_fid /kaggle/working/real_images /kaggle/working/fake_images

print("\n📦 Packaging final files...")
os.chdir('/kaggle/working')
!zip -q -r evaluation_results.zip fake_images/ real_images/
print("✅ Done! Download 'evaluation_results.zip'.")